# 06 · XGBoost regression for early-onset CRC

**Question:** can tuned XGBoost improve prediction of early-onset (ages 20–49) colorectal cancer incidence compared with the existing models?

**Outcome:** state-year crude CRC incidence rate per 100,000 adults ages 20–49.

**Features:** the same predictors as Notebook 3: year, state baseline early-onset rate, red meat and poultry availability, adult smoking, adult obesity, uninsured rate, log population, and Census region.

**Validation:**
- **5-fold GroupKFold by state**, with parameter tuning inside each training fold.
- **Temporal holdout 2017–2019:** train through 2016, test 2017–2019.
- **Temporal holdout 2020:** train through 2019, test 2020.
- **Combined 2017–2020 holdout:** train through 2016, predict all four years without refitting.

**Model:** XGBoost regression (boosted decision trees). The results discussed below are from the completed project run; rerunning the code will recalculate them. This notebook does not change the original models or datasets.

In [ ]:
import os, sys, tempfile
from pathlib import Path


def _find_project_root(start=None):
    """Locate the project folder containing src/data_prep.py."""
    here = Path(start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "data_prep.py").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Open this notebook inside the full repository "
        "(the folder containing src/, notebooks/, and outputs/)."
    )

ROOT = _find_project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "mpl"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import GroupKFold, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor
import xgboost

from src.model import NUMERIC, CATEGORICAL, TARGET

CLEAN = ROOT / "outputs" / "cleaned"
FIGURES = ROOT / "figures"
CLEAN.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print(f"Project root: {ROOT}")
print(f"XGBoost version: {xgboost.__version__}")

## 1. Prepare the data

The outcome and predictor list match the original `03_modeling.ipynb` via `src.model`. Missing numeric predictor values are imputed **within each training fold**, and Census region is one-hot encoded. XGBoost predicts the cancer **rate**, not the number of cases.

In [ ]:
path = CLEAN / "analysis_panel_1999_2020.csv"
if not path.exists():
    raise FileNotFoundError(f"Missing {path}. Run 01_data_cleaning.ipynb first.")

df = pd.read_csv(path).dropna(subset=[TARGET]).copy().reset_index(drop=True)
df["region"] = df["region"].fillna("Unknown")
X = df[NUMERIC + CATEGORICAL].copy()
y = df[TARGET].copy()
groups = df["state_abbr"]

print(f"Modeling rows: {len(df):,}; states: {groups.nunique()}")
print("Predictors:", NUMERIC + CATEGORICAL)
print("Years:", int(df.year.min()), "to", int(df.year.max()))

## 2. XGBoost and tuning approach

XGBoost fits a sequence of decision trees to capture nonlinear relationships and interactions between predictors. We tested a small set of configurations with different tree depths, learning rates, numbers of trees, and regularization settings.

Hyperparameters are selected using **MAE** on training-period validation data. For GroupKFold, each outer training fold has a separate inner state-grouped tuning step. For temporal holdouts, only earlier years are used for tuning. The held-out states or years are not used to select parameters.

In [ ]:
# Small candidate set: raise N_JOBS to speed up on a machine with more cores.
N_JOBS = 2
CANDIDATES = [
    dict(max_depth=2, n_estimators=180, learning_rate=0.03, min_child_weight=5,
         subsample=0.85, colsample_bytree=0.85, reg_lambda=5),
    dict(max_depth=2, n_estimators=300, learning_rate=0.03, min_child_weight=5,
         subsample=0.90, colsample_bytree=0.90, reg_lambda=10),
    dict(max_depth=3, n_estimators=180, learning_rate=0.05, min_child_weight=3,
         subsample=0.85, colsample_bytree=0.85, reg_lambda=5),
    dict(max_depth=3, n_estimators=300, learning_rate=0.03, min_child_weight=3,
         subsample=0.90, colsample_bytree=0.90, reg_lambda=10),
    dict(max_depth=4, n_estimators=160, learning_rate=0.05, min_child_weight=5,
         subsample=0.85, colsample_bytree=0.85, reg_lambda=10),
]


def make_xgb(params, numeric=NUMERIC, categorical=CATEGORICAL):
    """Fully train-fold-fitted feature processing and XGBoost regression."""
    prep = ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), list(numeric)),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), list(categorical)),
    ], remainder="drop")
    model = XGBRegressor(
        objective="reg:squarederror", tree_method="hist", random_state=42,
        n_jobs=N_JOBS, verbosity=0, **params
    )
    return Pipeline([("prep", prep), ("model", model)])


def choose_by_group_cv(X_train, y_train, group_train, n_splits=3):
    """Tune on *training-only* states using mean validation MAE."""
    rows = []
    cv = GroupKFold(n_splits=n_splits)
    splits = list(cv.split(X_train, y_train, groups=group_train))
    for ix, params in enumerate(CANDIDATES):
        errors = []
        for tr, val in splits:
            m = make_xgb(params)
            m.fit(X_train.iloc[tr], y_train.iloc[tr])
            errors.append(mean_absolute_error(y_train.iloc[val], m.predict(X_train.iloc[val])))
        rows.append({"candidate": ix, "validation_mae": float(np.mean(errors))})
    scores = pd.DataFrame(rows).sort_values("validation_mae")
    best_id = int(scores.iloc[0]["candidate"])
    return best_id, scores


def choose_by_past_years(train_df, X_train, y_train):
    """Tune on an older/past segment of the available training period only."""
    last_year = int(train_df.year.max())
    # Example: <=2012 -> 2013-2016 for the 2017-2019 test.
    valid_start = last_year - 3
    tr = train_df.year < valid_start
    val = train_df.year >= valid_start
    if tr.sum() == 0 or val.sum() == 0:
        raise ValueError("Insufficient years for the temporal tuning split.")
    rows = []
    for ix, params in enumerate(CANDIDATES):
        m = make_xgb(params)
        m.fit(X_train.loc[tr], y_train.loc[tr])
        pred = m.predict(X_train.loc[val])
        rows.append({"candidate": ix, "validation_mae": mean_absolute_error(y_train.loc[val], pred)})
    scores = pd.DataFrame(rows).sort_values("validation_mae")
    best_id = int(scores.iloc[0]["candidate"])
    return best_id, scores, valid_start


def metric_row(validation, truth, prediction):
    return {
        "model": "XGBoost regression",
        "validation": validation,
        "r2": r2_score(truth, prediction),
        "mae": mean_absolute_error(truth, prediction),
        "n": len(truth),
    }

## 3. Model performance

We use the same three evaluation settings as Notebook 3. **R²** measures how much variation in incidence the model explains; **MAE** is the average prediction error in cases per 100,000. Higher R² and lower MAE are better.

In [ ]:
rows = []
oof_pred = np.full(len(df), np.nan)
tune_log = []

# Nested GroupKFold: do not tune on the held-out outer states.
outer_cv = GroupKFold(n_splits=5)
for fold, (tr, val) in enumerate(outer_cv.split(X, y, groups=groups), start=1):
    best_id, scores = choose_by_group_cv(X.iloc[tr], y.iloc[tr], groups.iloc[tr])
    fitted = make_xgb(CANDIDATES[best_id])
    fitted.fit(X.iloc[tr], y.iloc[tr])
    oof_pred[val] = fitted.predict(X.iloc[val])
    tune_log.append({"evaluation": f"GroupKFold fold {fold}", "candidate": best_id,
                     "inner_mae": float(scores.iloc[0]["validation_mae"])})
    print(f"Outer fold {fold}/5: candidate {best_id}; inner MAE {scores.iloc[0]['validation_mae']:.3f}")

rows.append(metric_row("5-fold GroupKFold (by state)", y, oof_pred))

# The same holdout periods as Notebook 3.
for label, train_mask, test_mask in [
    ("Temporal holdout 2017-2019", df.year <= 2016, df.year.between(2017, 2019)),
    ("Temporal holdout 2020", df.year <= 2019, df.year == 2020),
]:
    train_df = df.loc[train_mask]
    best_id, scores, valid_start = choose_by_past_years(
        train_df, X.loc[train_mask], y.loc[train_mask]
    )
    fitted = make_xgb(CANDIDATES[best_id])
    fitted.fit(X.loc[train_mask], y.loc[train_mask])
    predicted = fitted.predict(X.loc[test_mask])
    rows.append(metric_row(label, y.loc[test_mask], predicted))
    tune_log.append({"evaluation": label, "candidate": best_id,
                     "inner_mae": float(scores.iloc[0]["validation_mae"])})
    print(f"{label}: candidate {best_id} tuned using years {valid_start}-{train_df.year.max()}")

xgb_metrics = pd.DataFrame(rows)
xgb_group_preds = df[["state_abbr", "year", TARGET]].copy()
xgb_group_preds["prediction"] = oof_pred
xgb_tuning = pd.DataFrame(tune_log)

display(xgb_metrics.round(3))
display(xgb_tuning.round(3))
xgb_metrics.round(6).to_csv(CLEAN / "xgboost_model_metrics.csv", index=False)
xgb_group_preds.to_csv(CLEAN / "xgboost_groupcv_predictions.csv", index=False)
xgb_tuning.to_csv(CLEAN / "xgboost_tuning_results.csv", index=False)

**Reading the metrics.** XGBoost achieved **R² = 0.531** and **MAE = 1.033** in five-fold GroupKFold. For the 2017–2019 temporal holdout, R² fell to **0.437** (MAE **1.048**). The separate 2020 holdout had R² **0.214** (MAE **1.168**). The lower temporal scores suggest that forecasting later years is harder than predicting rates for held-out states. The 2020 evaluation includes data through 2019 in its training set.

## 4. Comparison with the other models

The comparison uses the saved results from Notebook 3 and the optional GAM and Poisson notebooks when their metric files are present. The models did not all use identical tuning procedures, so the comparison is descriptive rather than a controlled algorithm benchmark.

In [ ]:
comparison = [xgb_metrics]
for name in ["model_metrics.csv", "gam_model_metrics.csv", "poisson_model_metrics.csv"]:
    file = CLEAN / name
    if file.exists():
        previous = pd.read_csv(file)
        comparison.append(previous[["model", "validation", "r2", "mae", "n"]])
    else:
        print(f"Optional comparison file not found: {name}")

all_metrics = pd.concat(comparison, ignore_index=True)
all_metrics = all_metrics.drop_duplicates(subset=["model", "validation"], keep="first")
print("R² comparison (higher is better):")
display(all_metrics.pivot(index="model", columns="validation", values="r2").round(3))
print("MAE comparison (lower is better; cases per 100,000):")
display(all_metrics.pivot(index="model", columns="validation", values="mae").round(3))

**Model comparison.** HGB had slightly better temporal R² than XGBoost, although the two boosting approaches performed similarly. In 2017–2019, HGB achieved **R² = 0.448** versus **0.437** for XGBoost. XGBoost had the lowest MAE in that period (**1.048**), narrowly ahead of GAM (**1.049**), random forest (**1.052**), and HGB (**1.053**). For 2020, HGB again performed better (**R² = 0.245, MAE = 1.149**) than XGBoost (**R² = 0.214, MAE = 1.168**). These differences are small, and HGB remains the preferred model on temporal R².

The optional Poisson comparison file was not found in this run. Its separately reported scores were GroupKFold R² **0.568**, 2017–2019 R² **0.190**, and 2020 R² **−0.354**.

## 5. Combined 2017–2020 holdout

For the report figure, the model is trained through **2016** and predicts 2017–2020 without refitting. This differs from the separate 2020 evaluation, which uses training data through 2019.

The scatterplot compares observed and predicted state-year incidence rates. The annual plot compares average observed and predicted incidence. Neither graph measures causal effects.

In [ ]:
train_mask = df.year <= 2016
test_mask = df.year.between(2017, 2020)

best_id, tuning_scores, valid_start = choose_by_past_years(
    df.loc[train_mask], X.loc[train_mask], y.loc[train_mask]
)
xgb_final = make_xgb(CANDIDATES[best_id])
xgb_final.fit(X.loc[train_mask], y.loc[train_mask])
holdout_pred = xgb_final.predict(X.loc[test_mask])

xgb_holdout = df.loc[test_mask, ["state_abbr", "year", TARGET]].copy()
xgb_holdout["prediction"] = holdout_pred
xgb_holdout["residual"] = xgb_holdout[TARGET] - xgb_holdout["prediction"]
xgb_holdout.to_csv(CLEAN / "xgboost_test_predictions.csv", index=False)

combined = metric_row("Combined temporal holdout 2017-2020", y.loc[test_mask], holdout_pred)
print(f"Chosen configuration: candidate {best_id} (tuned on {valid_start}-2016)")
print(f"2017–2020 holdout R²: {combined['r2']:.3f}")
print(f"2017–2020 holdout MAE: {combined['mae']:.3f} cases per 100,000")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
ax = axes[0]
sc = ax.scatter(xgb_holdout[TARGET], xgb_holdout["prediction"],
                c=xgb_holdout.year, cmap="viridis", s=25, alpha=0.80)
lo = min(xgb_holdout[TARGET].min(), xgb_holdout.prediction.min()) - .5
hi = max(xgb_holdout[TARGET].max(), xgb_holdout.prediction.max()) + .5
ax.plot([lo, hi], [lo, hi], linestyle="--", color="gray", linewidth=1)
ax.set(xlabel="Observed CRC rate per 100,000", ylabel="Predicted CRC rate per 100,000",
       title="a. Observed vs predicted (2017–2020)", xlim=(lo, hi), ylim=(lo, hi))
fig.colorbar(sc, ax=ax, label="Year")

yearly = xgb_holdout.groupby("year")[[TARGET, "prediction"]].mean()
ax = axes[1]
ax.plot(yearly.index, yearly[TARGET], marker="o", label="Observed")
ax.plot(yearly.index, yearly["prediction"], marker="s", linestyle="--", label="XGBoost prediction")
ax.set(xlabel="Year", ylabel="Mean state-level rate per 100,000",
       title="b. Mean observed and predicted rate by year")
ax.set_xticks(list(yearly.index))
ax.legend()
for ax in axes:
    ax.grid(alpha=0.2)
fig.tight_layout()
fig_path = FIGURES / "fig7_xgboost_holdout.png"
fig.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print("Saved:", fig_path)

**Figure interpretation.** The observed-versus-predicted plot shows that XGBoost captures some differences among states, but predictions are compressed toward the middle: high observed cancer rates are often underestimated, while low rates are often overestimated. The annual mean predictions are nearly flat and below observed incidence; the observed series rises through 2019 and dips in 2020.

For the **combined 2017–2020 holdout**, the model achieved **R² = 0.354** and **MAE = 1.099** cases per 100,000. This score is not directly comparable to the separate 2020 score because the combined holdout never refits after 2016.

## 6. Feature importance and meat availability

Permutation importance measures how much the **2017–2020 holdout R²** decreases after a predictor is shuffled. We also compare the model with and without red meat and poultry availability to check whether these variables improve holdout prediction. Both analyses concern **predictive value**, not causation.

In [ ]:
perm = permutation_importance(
    xgb_final, X.loc[test_mask], y.loc[test_mask], scoring="r2",
    n_repeats=15, random_state=42, n_jobs=1,
)
xgb_importance = (
    pd.DataFrame({
        "feature": X.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
xgb_importance.to_csv(CLEAN / "xgboost_permutation_importance.csv", index=False)
display(xgb_importance.round(3))

fig, ax = plt.subplots(figsize=(7, 4))
show = xgb_importance.sort_values("importance_mean")
ax.barh(show.feature, show.importance_mean, xerr=show.importance_std, capsize=2)
ax.axvline(0, color="gray", linewidth=1)
ax.set(xlabel="Drop in holdout R² after permutation", title="XGBoost: 2017–2020 holdout importance")
fig.tight_layout()
fig.savefig(FIGURES / "fig8_xgboost_importance.png", dpi=200, bbox_inches="tight")
plt.show()

no_meat_features = [c for c in NUMERIC if c not in ["red_meat_lb_pc", "poultry_lb_pc"]]
xgb_no_meat = make_xgb(CANDIDATES[best_id], numeric=no_meat_features, categorical=CATEGORICAL)
xgb_no_meat.fit(X.loc[train_mask], y.loc[train_mask])
no_meat_pred = xgb_no_meat.predict(X.loc[test_mask])
feature_comparison = pd.DataFrame([
    metric_row("Combined 2017–2020: with meat", y.loc[test_mask], holdout_pred),
    metric_row("Combined 2017–2020: without meat", y.loc[test_mask], no_meat_pred),
])
display(feature_comparison[["validation", "r2", "mae"]].round(3))

**Feature importance.** Baseline CRC incidence was the dominant predictor (permutation importance **0.261**), followed by obesity (**0.100**), smoking (**0.048**), and log population (**0.033**). Region had little importance, while year, red meat, and poultry were approximately zero in the temporal holdout. Tree models cannot extrapolate the year trend beyond the years seen during training, so zero year importance should not be taken as evidence that time trends do not matter.

**Incremental value of meat.** The combined holdout gave **R² = 0.354, MAE = 1.099** with meat availability and **R² = 0.341, MAE = 1.113** without it. Including meat improved R² by **0.013** and reduced MAE by **0.014** cases per 100,000. The improvement is modest and does not show that red meat consumption causes changes in CRC incidence.

## 7. Interpretation and limitations

- **Model selection.** XGBoost was competitive with HGB but did not outperform it on temporal R². HGB scored **0.448 versus 0.437** in 2017–2019 and **0.245 versus 0.214** in 2020. XGBoost had a slightly lower 2017–2019 MAE (**1.048 versus 1.053**), a difference too small to treat as a clear advantage.
- **Generalization.** GroupKFold evaluates held-out states, whereas temporal holdouts evaluate future years. XGBoost performed more strongly on unseen states than on future-year changes.
- **Main predictors.** The state's baseline early-onset CRC rate had the largest holdout permutation importance, followed by obesity, smoking, and population. This identifies features useful to prediction, not causal risk-factor effects.
- **Meat availability.** Red meat and poultry had near-zero individual permutation importance. Nevertheless, the combined holdout R² was **0.354 with meat versus 0.341 without meat**. This small improvement does not establish a dietary effect; removing and refitting predictors tests something different from shuffling a predictor in a fixed model.
- **Historical baseline.** As in Notebook 3, `crc_early_rate_baseline` is calculated from the outcome. Its use in state-grouped validation may be overly optimistic, particularly when a held-out state's baseline is unavailable at prediction time. This should be revisited with the group.
- **Exposure measurement.** USDA meat availability is national, not state-specific consumption, and does not isolate processed meat. National time trends may be confounded with other changes in cancer diagnosis and risk factors.
- **Model comparison.** The candidate XGBoost configurations were limited, and other models had different tuning procedures. Small differences in MAE or R² should not be overstated.

**Conclusion:** XGBoost is a competitive additional model but does not replace HGB as the strongest temporal R² performer in this project. Its clearest finding is that baseline cancer burden, obesity, and smoking help distinguish predicted rates, while national meat availability adds only modest incremental predictive value. These are associations, not evidence of causation.